# 18 — Azure AI Foundry og sky-distribusjon

**Fase:** 4 — Sky & DevOps | **Tid:** 2 timer | **Krav:** Notatbok 05, 17

**Hva du bygger:** Forståelse av Azure AI Foundry-arkitekturen, og Python-kode som er klar for distribusjon til Azure — uten å kreve en Azure-konto for å kjøre koden.

---

## Gratis tilgang til Azure

- **Azure gratis konto:** 200 USD i gratis kreditter i 30 dager (azure.microsoft.com/free)
- **GitHub Student Pack:** Ekstra Azure-kreditter for studenter
- **Koden i denne notatboken** kjøres mot Ollama lokalt — bytt bare base_url for Azure

---

## Azure AI Foundry — Arkitektur

```
Azure AI Foundry
├── AI Hub            ← Delt infrastruktur (tilgang, nøkler, nettverk)
│   ├── AI Project A  ← SPK-pensjonsassistent
│   └── AI Project B  ← Avinor-flyassistent
├── Model Catalog     ← GPT-4o, Llama, Mistral, Phi osv.
├── Azure OpenAI      ← Distribuert GPT-4o endpoint
└── AI Search         ← Vektorsøk + hybrid søk
```

**Nøkkelbegreper:**
- **Deployment** — En instans av en modell med gitt kapasitet
- **PTU** (Provisioned Throughput Units) — Reservert kapasitet, forutsigbar ytelse
- **Pay-as-you-go** — Betal per token, lavere kostnad ved lav trafikk
- **Managed Identity** — Ingen passordet i kode, Azure håndterer autentisering

In [ ]:
%pip install -q openai python-dotenv azure-identity

---

## Del 1: Konfigurasjonsoppsett

God praksis: Skill ut konfigurasjon fra kode. Apper bør fungere mot Ollama lokalt og Azure i sky ved å bytte én config.

In [ ]:
import os
from dataclasses import dataclass
from openai import OpenAI

@dataclass
class LLMKonfig:
    leverandør:  str
    base_url:    str
    api_key:     str
    modell:      str

def hent_konfig() -> LLMKonfig:
    """Les konfig fra miljøvariabler, fall tilbake til Ollama."""
    leverandør = os.getenv("LLM_LEVERANDOR", "ollama")

    if leverandør == "azure":
        return LLMKonfig(
            leverandør="azure",
            base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
            api_key=os.environ["AZURE_OPENAI_KEY"],
            modell=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
        )
    elif leverandør == "groq":
        return LLMKonfig(
            leverandør="groq",
            base_url="https://api.groq.com/openai/v1",
            api_key=os.environ["GROQ_API_KEY"],
            modell="llama-3.3-70b-versatile",
        )
    else:  # Ollama standard
        return LLMKonfig(
            leverandør="ollama",
            base_url="http://localhost:11434/v1",
            api_key="ollama",
            modell="llama3.2",
        )

konfig = hent_konfig()
print(f"Bruker: {konfig.leverandør} / {konfig.modell}")

client = OpenAI(base_url=konfig.base_url, api_key=konfig.api_key)

In [ ]:
# Test tilkoblingen
svar = client.chat.completions.create(
    model=konfig.modell,
    messages=[{"role": "user", "content": "Si 'tilkobling OK' på norsk"}],
    max_tokens=20,
)
print(svar.choices[0].message.content)

---

## Del 2: Managed Identity — Ingen passord i koden

I produksjon på Azure bruker du **Managed Identity** i stedet for API-nøkler:

In [ ]:
# Eksempel: Slik kobler du til Azure OpenAI med Managed Identity
# (Krever Azure — dette er referansekode, ikke kjørbar lokalt)

azure_managed_identity_kode = """
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# DefaultAzureCredential prøver: Managed Identity → Azure CLI → VS Code → osv.
# Ingen hardkodede nøkler!
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01",
)

# Bruk nøyaktig som vanlig OpenAI-klient
svar = client.chat.completions.create(
    model="gpt-4o",  # Deployment-navn i Azure
    messages=[{"role": "user", "content": "Hei"}]
)
"""

print("Managed Identity-kode (referanse):")
print(azure_managed_identity_kode)

---

## Del 3: Koste- og kapasitetsmodell

In [ ]:
# Hjelp til å velge mellom PTU og Pay-as-you-go

@dataclass
class AzureKostnad:
    kall_per_dag:        int
    input_tokens_pr_kall: int
    output_tokens_pr_kall: int

def beregn_månedskostnad(bruk: AzureKostnad) -> dict:
    # GPT-4o priser (omtrentlig, sjekk azure.com/pricing for oppdaterte priser)
    PAYG_INPUT  = 0.0025 / 1000  # USD per token
    PAYG_OUTPUT = 0.010  / 1000

    daglig_input  = bruk.kall_per_dag * bruk.input_tokens_pr_kall
    daglig_output = bruk.kall_per_dag * bruk.output_tokens_pr_kall

    månedlig_payg = 30 * (
        daglig_input  * PAYG_INPUT +
        daglig_output * PAYG_OUTPUT
    )

    return {
        "kall_per_måned":   bruk.kall_per_dag * 30,
        "tokens_per_måned": (daglig_input + daglig_output) * 30,
        "payg_usd":         round(månedlig_payg, 2),
        "anbefaling":       "PTU" if bruk.kall_per_dag > 5000 else "Pay-as-you-go",
    }

scenarier = [
    ("Intern SPK-chatbot (lav)",  AzureKostnad(200,  500, 300)),
    ("Pensjonsveiledning (middels)", AzureKostnad(2000, 800, 500)),
    ("Offentlig nett-chatbot (høy)", AzureKostnad(10000, 600, 400)),
]

print(f"{'Scenario':<35} {'Kall/mnd':>10} {'PAYG (USD)':>12} {'Anbefaling':>15}")
print("-" * 75)
for navn, bruk in scenarier:
    r = beregn_månedskostnad(bruk)
    print(f"{navn:<35} {r['kall_per_måned']:>10,} {r['payg_usd']:>12.2f} {r['anbefaling']:>15}")

---

## Del 4: Infrastruktur som kode (IaC)

Avinor-annonsen nevner **IaC**. Her er en Bicep-template for å opprette Azure OpenAI:

In [ ]:
bicep_template = """
// azure_openai.bicep — opprett Azure OpenAI med Bicep (IaC)
param location string = resourceGroup().location
param aiServiceNavn string = 'spk-openai'

resource openAI 'Microsoft.CognitiveServices/accounts@2023-05-01' = {
  name: aiServiceNavn
  location: location
  kind: 'OpenAI'
  sku: { name: 'S0' }
  properties: {
    publicNetworkAccess: 'Disabled'       // Privat nettverkstilgang
    customSubDomainName: aiServiceNavn
  }
}

resource deployment 'Microsoft.CognitiveServices/accounts/deployments@2023-05-01' = {
  parent: openAI
  name: 'gpt-4o'
  properties: {
    model: { format: 'OpenAI', name: 'gpt-4o', version: '2024-05-13' }
  }
  sku: { name: 'Standard', capacity: 10 }  // 10k tokens/minutt
}

output endpoint string = openAI.properties.endpoint
"""

# Distribuer med: az deployment group create --template-file azure_openai.bicep
print("Bicep-template (IaC for Azure OpenAI):")
print(bicep_template)

---

## Oppsummering

| Konsept | Hva det er |
|---------|----------|
| AI Foundry | Sentralt sted for AI-prosjekter i Azure |
| AI Hub | Delt infrastruktur for flere prosjekter |
| Managed Identity | Autentisering uten passord i koden |
| PTU | Reservert kapasitet for produksjon med mye trafikk |
| Pay-as-you-go | Betal per token, bra for lavt/ujevnt volum |
| Bicep / IaC | Infrastruktur definert som kode, reproduserbart |

---

## Hva er neste steg?

**Siste notatbok: `19_ai_security_gdpr.ipynb`** — Ansvarlig AI, GDPR for RAG-systemer, prompt injection og EU AI Act. Avslutter læringsplanen.